<a href="https://colab.research.google.com/github/EgesaR/training-hub/blob/main/speech_stt_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torchaudio matplotlib

In [ ]:
# # Clone
# !git clone https://github.com/EgesaR/training-hub.git
# %cd training-hub

In [ ]:
# !git add .
# !git commit -m "update training loop"
# !git push

# TinyASR v1 — Robust, Prosody-Aware Speech Recognition

## What v1 Is

**TinyASR v1** is the foundational version of a lightweight, CPU-friendly automatic speech recognition system designed to recognize a controlled vocabulary of spoken words while remaining robust to silence, background noise, and varying recording conditions.

The primary objective of v1 is to establish a **reliable, measurable, and extensible baseline** before introducing more advanced architectures and calibration techniques in later versions.

v1 focuses on:

* Reliable audio preprocessing
* Log-Mel speech representations
* Basic acoustic/prosody analysis
* Noise and silence handling
* Lightweight neural-network inference
* Basic confidence estimation
* Basic abstention
* Efficient CPU/GPU training
* Feature caching
* Checkpointing and resume
* Reproducible evaluation
* A modular architecture that can evolve into v2 and v3

---

# 1. Core Goals

TinyASR v1 should achieve the following:

### Speech Recognition

* Recognize words from a defined vocabulary.
* Support the Speech Commands dataset and similar datasets.
* Operate efficiently on CPU.
* Support GPU acceleration when available.
* Produce predictions with confidence values.
* Avoid forcing every audio input into a valid word.

### Robustness

The model should be exposed to:

* Clean speech
* Background noise
* Silence
* Different speakers
* Different accents
* Different recording conditions
* Small timing variations
* Different speaking volumes

### Acoustic Analysis

In addition to recognizing words, v1 should extract measurable properties of the input audio, including:

* Pitch
* Energy
* Voicing
* Duration
* Pitch range
* Pitch direction
* Spectral brightness

These measurements form the foundation for more advanced prosody and tone analysis in later versions.

### Engineering

The system should be:

* Lightweight
* Modular
* Reproducible
* Cache-friendly
* Checkpointable
* Resumable
* CPU-aware
* GPU-aware
* Easy to benchmark
* Easy to extend

---

# 2. v1 Feature Set

| #  | Feature                         | v1 | Purpose                             |
| -- | ------------------------------- | -: | ----------------------------------- |
| 1  | Log-Mel Spectrogram             |  ✅ | Primary speech representation       |
| 2  | Basic Prosody Extraction        |  ✅ | Capture acoustic characteristics    |
| 3  | Acoustic/Tone Descriptors       |  ✅ | Human-readable audio analysis       |
| 4  | `<noise>` Class                 |  ✅ | Handle silence and background noise |
| 5  | Lightweight Encoder             |  ✅ | Efficient inference                 |
| 6  | Word Classification Head        |  ✅ | Predict vocabulary words            |
| 7  | Basic Confidence Score          |  ✅ | Estimate prediction certainty       |
| 8  | Basic Abstention                |  ✅ | Reject weak predictions             |
| 9  | Audio Augmentation              |  ✅ | Improve robustness                  |
| 10 | SpecAugment                     |  ✅ | Improve feature robustness          |
| 11 | Feature Cache                   |  ✅ | Reduce preprocessing cost           |
| 12 | CPU/GPU Detection               |  ✅ | Adapt to available hardware         |
| 13 | Checkpoint Saving               |  ✅ | Preserve training progress          |
| 14 | Training Resume                 |  ✅ | Recover interrupted training        |
| 15 | Validation Evaluation           |  ✅ | Monitor generalization              |
| 16 | Confusion Matrix                |  ✅ | Analyze recognition errors          |
| 17 | Noise Rejection Evaluation      |  ✅ | Measure non-speech handling         |
| 18 | Modular Configuration           |  ✅ | Make experiments reproducible       |
| 19 | CTC Head                        |  ❌ | Planned for v2                      |
| 20 | Attention Classifier            |  ❌ | Planned for v2                      |
| 21 | CTC + Classifier Fusion         |  ❌ | Planned for v2                      |
| 22 | Temperature Scaling             |  ❌ | Planned for v2/v3                   |
| 23 | Advanced Abstention Calibration |  ❌ | Planned for v2/v3                   |
| 24 | EMA Weights                     |  ❌ | Planned for v2                      |
| 25 | Advanced Prosody Fusion         |  ❌ | Planned for v2/v3                   |
| 26 | Emotion Recognition             |  ❌ | Requires separate labeled data      |
| 27 | Intent Recognition              |  ❌ | Requires separate labeled data      |

---

# 3. Audio Frontend

## 3.1 Audio Input

The initial frontend should normalize incoming audio into a consistent format.

Target configuration:

```text
Sample Rate:       16,000 Hz
Channels:          Mono
Audio Format:      Float32
Target Duration:   ~1 second
```

The frontend should handle:

* Different input sample rates
* Stereo → mono conversion
* Amplitude normalization
* Padding
* Trimming
* Short audio
* Long audio
* Invalid/empty input

---

# 4. Log-Mel Spectrogram

The primary representation in v1 is the **log-Mel spectrogram**.

Pipeline:

```text
Raw Audio
    │
    ▼
Resampling
    │
    ▼
Mono Conversion
    │
    ▼
Amplitude Normalization
    │
    ▼
STFT
    │
    ▼
Mel Filter Bank
    │
    ▼
Log Compression
    │
    ▼
Log-Mel Spectrogram
```

The resulting representation becomes the primary input to the neural network.

Example conceptual shape:

```text
Audio
  │
  └──> [Mel Frequency × Time]
                │
                ▼
            Encoder
```

The exact number of Mel bins, FFT size, hop length, and window size should remain configurable.

Example configuration:

```python
SAMPLE_RATE = 16_000

N_FFT = 400
HOP_LENGTH = 160
N_MELS = 80

F_MIN = 20
F_MAX = 7_600
```

These values are experimental defaults rather than permanent requirements.

---

# 5. Basic Prosody Features

v1 extracts basic acoustic features alongside the log-Mel representation.

The purpose is initially **measurement and analysis**, rather than claiming that these features automatically improve recognition.

## 5.1 Energy

Energy provides information about the amplitude/loudness characteristics of speech.

Possible representation:

```text
Energy(t)
```

It can be used to estimate:

* Relative loudness
* Speech activity
* Peaks
* Energy variation

---

## 5.2 Pitch / Fundamental Frequency

Estimate the fundamental frequency:

```text
F0(t)
```

This provides information about vocal pitch.

Useful descriptors include:

```text
Mean F0
Minimum F0
Maximum F0
Pitch Range
Pitch Slope
```

Unvoiced frames should be handled separately rather than treating missing F0 values as real pitch measurements.

---

## 5.3 Voicing

Estimate whether individual frames contain voiced speech.

Conceptually:

```text
Frame → voiced / unvoiced
```

This enables calculation of:

```text
Voiced Ratio =
voiced frames / speech frames
```

---

## 5.4 Duration

Measure:

```text
Total Audio Duration
Estimated Speech Duration
```

This allows v1 to distinguish between:

* Very short inputs
* Normal speech
* Long speech
* Silence

---

## 5.5 Additional Acoustic Features

v1 may also calculate:

* Zero-crossing rate
* Spectral centroid
* Spectral energy
* Spectral bandwidth

These features are primarily intended for analysis and future experimentation.

---

# 6. Utterance Acoustic Descriptors

Every inference can optionally produce a human-readable acoustic report.

Example:

```text
Acoustic Analysis
-----------------
Speech Duration: 0.74 s
Voiced Ratio:    0.68
Mean Pitch:      184 Hz
Pitch Range:     92 Hz
Pitch Slope:     Rising
Mean Energy:     -18.4 dB
Peak Energy:     -4.8 dB
Brightness:      2.7 kHz
```

These values describe measurable properties of the recording.

They should **not** be interpreted as definitive emotional states.

For example:

```text
Pitch increased
```

is a measurable observation.

It should not automatically become:

```text
The speaker is excited.
```

---

# 7. Speech / No-Speech Handling

One of the important goals of v1 is to prevent the model from treating silence or random noise as a valid word.

The vocabulary therefore includes:

```text
<noise>
```

The `<noise>` category may contain examples such as:

* Silence
* Background noise
* Environmental sounds
* Non-target audio
* Very weak speech
* Unusable recordings

Conceptually:

```text
Input Audio
     │
     ▼
   Model
     │
     ├──> "yes"
     ├──> "no"
     ├──> "up"
     ├──> "down"
     ├──> ...
     └──> "<noise>"
```

This provides the model with an explicit alternative to incorrectly selecting a vocabulary word.

---

# 8. Vocabulary

The model should operate within a clearly defined vocabulary.

For a full Speech Commands configuration, the supported vocabulary can include the 35 standard target words.

The vocabulary should be represented through a deterministic mapping:

```python
label_to_id = {
    "<noise>": 0,
    "backward": 1,
    "bed": 2,
    "bird": 3,
    ...
}
```

And the reverse mapping:

```python
id_to_label = {
    0: "<noise>",
    1: "backward",
    2: "bed",
    3: "bird",
    ...
}
```

The mapping must be saved with the model checkpoint.

---

# 9. Model Architecture

v1 intentionally uses a relatively simple architecture.

The basic design is:

```text
              Audio
                │
                ▼
        Log-Mel Spectrogram
                │
                ▼
        Lightweight Encoder
                │
                ▼
       Global/Temporal Pool
                │
                ▼
       Classification Head
                │
                ▼
             Softmax
                │
                ▼
          Word / <noise>
```

The encoder can be implemented using a lightweight CNN or another compact temporal architecture.

The design priority is:

```text
Accuracy
   +
Robustness
   +
Low CPU cost
   +
Simple training
```

rather than maximum architectural complexity.

---

# 10. Classification Head

The final classification layer produces logits:

```text
logits = Model(x)
```

Then probabilities:

```text
probabilities = softmax(logits)
```

The predicted class is:

```text
prediction = argmax(probabilities)
```

Example:

```text
yes      0.91
no       0.03
up       0.02
down     0.01
<noise>  0.01
```

The predicted word is:

```text
yes
```

with a basic confidence value of:

```text
0.91
```

---

# 11. Basic Confidence

v1 provides a basic confidence score:

```python
confidence = probabilities.max()
```

This is useful for:

* Displaying prediction certainty
* Filtering very weak predictions
* Performing simple abstention
* Evaluating model behavior

However, v1 does **not** claim that:

```text
0.90 confidence = 90% probability of being correct
```

Softmax confidence can be poorly calibrated.

Proper calibration is therefore reserved for a later version.

---

# 12. Basic Abstention

v1 should have a configurable confidence threshold.

Example:

```python
CONFIDENCE_THRESHOLD = 0.70
```

Conceptually:

```python
if confidence < CONFIDENCE_THRESHOLD:
    prediction = "uncertain"
```

Therefore:

```text
Input
 │
 ▼
Model
 │
 ├── Confidence >= threshold
 │       │
 │       └──> Word
 │
 └── Confidence < threshold
         │
         └──> uncertain
```

If the model explicitly predicts `<noise>`, the output should remain `<noise>` rather than being converted into a vocabulary word.

---

# 13. "No Hallucination" Definition

TinyASR cannot guarantee that it will never make mistakes.

Instead, v1 follows a narrower and measurable principle:

> **The model should only output classes that exist in its vocabulary, or abstain when its basic confidence is insufficient.**

For example:

```text
Audio → "yes"
Audio → "down"
Audio → "<noise>"
Audio → "uncertain"
```

It should not generate arbitrary words outside the trained vocabulary.

This is **closed-vocabulary recognition**, not unrestricted speech generation.

---

# 14. Data Augmentation

v1 includes audio augmentation to improve robustness.

Potential augmentations:

### Time Shift

Move the speech slightly within the audio window.

```text
Before:
[ silence ][ speech ]

After:
[ silence ][   speech ]
```

### Background Noise

Mix speech with controlled background noise.

```text
clean speech
     +
noise
     ↓
noisy speech
```

### Gain Variation

Randomly vary amplitude to simulate different recording volumes.

### Time Masking

Mask portions of the feature representation.

### Frequency Masking

Mask frequency regions of the spectrogram.

---

# 15. SpecAugment

SpecAugment can be applied after feature extraction.

Conceptually:

```text
Log-Mel
  │
  ├── Frequency Mask
  │
  ├── Time Mask
  │
  └── Optional Time Warping
        │
        ▼
   Augmented Feature
```

The exact augmentation strength should be configurable because excessive augmentation can damage small datasets.

---

# 16. Feature Cache

Feature extraction can become a significant portion of training time.

v1 therefore supports caching.

Pipeline:

```text
Raw Dataset
     │
     ▼
Audio Preprocessing
     │
     ▼
Log-Mel / Acoustic Features
     │
     ▼
Feature Cache
     │
     ▼
Training
```

Instead of repeatedly performing:

```text
Load audio
→ resample
→ calculate STFT
→ calculate Mel
```

the training process can reuse cached features.

Example:

```text
cache/
├── train/
│   ├── sample_001.pt
│   ├── sample_002.pt
│   └── ...
├── validation/
└── test/
```

The cache should contain enough metadata to detect incompatible preprocessing configurations.

For example:

```text
sample_rate
n_fft
hop_length
n_mels
frontend_version
```

---

# 17. Dataset Splits

The dataset should be separated into:

```text
Training
Validation
Testing
```

The roles are:

### Training

Used to update model parameters.

### Validation

Used to:

* Monitor overfitting
* Select checkpoints
* Tune basic thresholds
* Compare experiments

### Test

Used only for final evaluation.

The test set should not be repeatedly used to tune the model.

---

# 18. Speaker Independence

Where supported by the dataset, evaluation should preserve speaker separation.

The goal is to avoid situations where:

```text
Speaker A → training
Speaker A → testing
```

creates an unrealistically easy evaluation.

A stronger evaluation setup is:

```text
Training speakers
        ≠
Testing speakers
```

This gives a better indication of generalization to unseen speakers.

---

# 19. Training

The v1 training loop should follow:

```text
Load Batch
    │
    ▼
Frontend / Cached Features
    │
    ▼
Model
    │
    ▼
Logits
    │
    ▼
Loss
    │
    ▼
Backpropagation
    │
    ▼
Optimizer
    │
    ▼
Validation
```

Basic metrics:

```text
Training Loss
Validation Loss
Training Accuracy
Validation Accuracy
Learning Rate
```

---

# 20. Loss Function

For the classification baseline, v1 can use cross-entropy loss:

```python
loss = CrossEntropyLoss(logits, labels)
```

Label smoothing may be evaluated later rather than making it mandatory in the baseline.

This makes the initial system easier to benchmark.

---

# 21. Optimizer

The optimizer should remain configurable.

For example:

```python
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
```

Configuration should be stored alongside the checkpoint.

---

# 22. Learning Rate Scheduling

v1 may use a simple learning-rate scheduler.

The exact scheduler should be selected experimentally.

Possible approaches:

```text
Cosine Decay
Step Decay
ReduceLROnPlateau
Warmup + Decay
```

The initial implementation should favor simplicity and reproducibility.

---

# 23. Hardware Awareness

TinyASR v1 should automatically detect the available compute device.

Conceptually:

```python
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
```

Future environments may include:

```text
CUDA GPU
CPU
Apple Silicon
Other accelerators
```

The training configuration should expose:

```text
Device
Batch Size
Workers
Precision
Gradient Accumulation
```

where supported.

---

# 24. CPU-Friendly Design

Because TinyASR is intended to run on resource-constrained hardware, v1 should track:

* Model parameter count
* Model size
* CPU inference latency
* Peak memory
* Feature extraction time
* End-to-end latency

Example benchmark:

```text
Model Parameters: 1.8M
Checkpoint Size:  7.4 MB
CPU Inference:    24 ms
Peak RAM:         180 MB
```

The actual values should be measured rather than assumed.

---

# 25. Checkpointing

Training should periodically save checkpoints.

Example:

```text
checkpoints/
├── last.pt
├── best.pt
└── epoch_010.pt
```

A checkpoint should contain:

```python
{
    "model_state_dict": ...,
    "optimizer_state_dict": ...,
    "scheduler_state_dict": ...,
    "epoch": ...,
    "global_step": ...,
    "best_metric": ...,
    "config": ...,
    "vocabulary": ...,
}
```

This makes the experiment reproducible and resumable.

---

# 26. Resume Training

If training is interrupted:

```text
Training
   │
   ├── Epoch 1
   ├── Epoch 2
   ├── Epoch 3
   └── Disconnect
          │
          ▼
       Resume
          │
          ├── Epoch 4
          ├── Epoch 5
          └── ...
```

The system should restore:

* Model weights
* Optimizer state
* Scheduler state
* Epoch
* Global step
* Best validation metric
* Relevant configuration

This is particularly important when training on temporary cloud environments.

---

# 27. Evaluation

v1 should generate a complete baseline evaluation.

## Overall Accuracy

```text
Correct predictions / Total predictions
```

## Per-Class Accuracy

Example:

```text
yes       94.2%
no        91.7%
up        88.3%
down      90.1%
...
```

## Confusion Matrix

A confusion matrix should identify commonly confused classes.

Example:

```text
          Predicted
          yes  no  up  down
Actual
yes        92   3   2    3
no          4  90   3    3
up          2   3  91    4
down        3   2   4   91
```

---

# 28. Noise Evaluation

The `<noise>` class should be evaluated separately.

Important measurements include:

```text
Noise correctly rejected
Noise incorrectly classified as a word
Silence incorrectly classified as a word
Weak speech rejected
```

This is important because high word accuracy alone does not guarantee good behavior on real-world audio.

---

# 29. False Activation Evaluation

A useful v1 metric is:

> How often does the model activate when no target word is present?

For example:

```text
1000 noise samples
↓
Model
↓
37 incorrectly classified as words
```

This gives:

```text
False Activation Rate = 3.7%
```

The exact metric definition should be documented in the evaluation code.

---

# 30. Inference Pipeline

The complete inference pipeline is:

```text
Microphone / Audio File
          │
          ▼
      Audio Loader
          │
          ▼
     Preprocessing
          │
          ▼
      Log-Mel
          │
          ├──────────────┐
          │              │
          ▼              ▼
       Model       Acoustic Analysis
          │              │
          ▼              ▼
      Probabilities   Descriptors
          │              │
          ▼              │
     Confidence         │
          │              │
          ▼              │
      Threshold          │
          │              │
          └──────┬───────┘
                 ▼
            Final Result
```

---

# 31. Example Inference Output

```text
TinyASR v1
==========

Prediction: yes
Confidence: 0.94

Speech:
  Duration: 0.73 s
  Voiced Ratio: 0.71

Acoustic:
  Mean Pitch: 182 Hz
  Pitch Range: 87 Hz
  Pitch Direction: Rising
  Mean Energy: -17.2 dB
  Peak Energy: -4.9 dB
  Brightness: 2.4 kHz
```

For uncertain input:

```text
TinyASR v1
==========

Prediction: uncertain
Confidence: 0.43

Reason:
  Confidence below threshold (0.70)

Acoustic:
  Speech Duration: 0.12 s
  Voiced Ratio: 0.08
```

For noise:

```text
TinyASR v1
==========

Prediction: <noise>
Confidence: 0.89
```

---

# 32. Tone Analysis in v1

v1 includes **acoustic analysis**, not emotion recognition.

The system can measure:

```text
Loudness
Pitch
Pitch range
Pitch direction
Duration
Voiced ratio
Brightness
Energy
```

It should report observations such as:

```text
Pitch: rising
Energy: high
Duration: short
Voicing: strong
```

It should not make unsupported claims such as:

```text
Emotion: angry
Intent: commanding
Mood: happy
```

Those require appropriately labeled datasets.

---

# 33. Emotion Recognition Is Not a v1 Feature

Speech Commands does not provide the necessary emotion labels for training an emotion classifier.

Therefore:

```text
Speech Commands
       │
       ├── Word Recognition ✅
       │
       └── Emotion Recognition ❌
```

Future emotion research could use datasets such as:

* RAVDESS
* CREMA-D
* Other appropriately licensed emotion datasets

This should be treated as a separate modeling problem.

---

# 34. Intent Recognition Is Not a v1 Feature

Intent requires semantic or intent-labeled speech data.

Speech Commands is primarily a controlled word-recognition dataset.

Therefore:

```text
"turn on the lights"
```

and

```text
"open the door"
```

represent a different problem from recognizing:

```text
"yes"
"no"
"up"
"down"
```

Future intent modeling would require sentence-level datasets such as Fluent Speech Commands or equivalent data.

---

# 35. What v1 Does NOT Try to Solve

v1 deliberately avoids:

* Large-vocabulary speech recognition
* Free-form transcription
* Emotion classification
* Intent classification
* Speaker identification
* Speaker verification
* Advanced conversational understanding
* CTC decoding
* Beam-search decoding
* Multi-head CTC/classification fusion
* Fully calibrated probabilities
* Advanced uncertainty estimation
* Learned emotion interpretation
* Semantic understanding

These are separate engineering/modeling problems.

---

# 36. Experiment Configuration

All important parameters should be centralized.

Example:

```python
CONFIG = {
    "sample_rate": 16_000,
    "n_fft": 400,
    "hop_length": 160,
    "n_mels": 80,

    "batch_size": 64,
    "epochs": 30,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,

    "confidence_threshold": 0.70,

    "use_prosody": True,
    "use_specaugment": True,
    "use_noise_class": True,

    "seed": 42,
}
```

The configuration should be saved into the checkpoint and experiment output.

---

# 37. Reproducibility

v1 should make experiments reproducible where practical.

Set:

```text
Random seed
Dataset split
Model configuration
Frontend configuration
Training configuration
Library versions
```

Each experiment should have a unique identifier.

Example:

```text
experiments/
└── tinyasr_v1_2026_09_21_001/
    ├── config.json
    ├── metrics.json
    ├── best.pt
    ├── last.pt
    └── evaluation.json
```

---

# 38. Recommended Notebook Structure

```text
TinyASR_v1.ipynb
│
├── 0. Setup
│   ├── Install dependencies
│   ├── Imports
│   ├── Configuration
│   ├── Random seeds
│   └── Device detection
│
├── 1. Text Helpers
│   ├── Vocabulary
│   ├── Label mapping
│   ├── Dataset utilities
│   └── Configuration helpers
│
├── 2. Audio Frontend
│   ├── Audio loading
│   ├── Resampling
│   ├── Mono conversion
│   ├── Normalization
│   ├── Log-Mel extraction
│   ├── Energy
│   ├── Pitch
│   ├── Voicing
│   ├── Spectral features
│   └── Acoustic descriptors
│
├── 3. Feature Cache
│   ├── Cache configuration
│   ├── Cache generation
│   ├── Cache validation
│   └── Cache statistics
│
├── 4. Load Splits
│   ├── Training dataset
│   ├── Validation dataset
│   ├── Test dataset
│   ├── Noise dataset
│   └── DataLoader
│
├── 5. Model
│   ├── Encoder
│   ├── Classification head
│   ├── Parameter count
│   └── Model summary
│
├── 6. Decoding
│   ├── Softmax
│   ├── Prediction
│   ├── Confidence
│   ├── Noise handling
│   └── Basic abstention
│
├── 7. Training
│   ├── Training loop
│   ├── Validation loop
│   ├── Metrics
│   ├── Checkpointing
│   ├── Resume
│   └── Training curves
│
├── 8. Evaluation
│   ├── Test accuracy
│   ├── Per-class accuracy
│   ├── Confusion matrix
│   ├── Noise rejection
│   ├── False activation
│   └── Error analysis
│
└── 9. Inference & Acoustic Analysis
    ├── Single-file inference
    ├── Batch inference
    ├── Prediction report
    ├── Confidence report
    ├── Acoustic descriptors
    └── Final demonstration
```

---

# 39. v1 Development Milestones

## Milestone 1 — Dataset

```text
Dataset
  ↓
Splits
  ↓
Vocabulary
  ↓
Noise samples
```

Deliverable:

```text
Reliable dataset pipeline
```

---

## Milestone 2 — Frontend

```text
Audio
  ↓
16 kHz mono
  ↓
Log-Mel
  ↓
Acoustic descriptors
```

Deliverable:

```text
Validated feature extraction pipeline
```

---

## Milestone 3 — Baseline Model

```text
Log-Mel
  ↓
Encoder
  ↓
Classifier
```

Deliverable:

```text
First working TinyASR model
```

---

## Milestone 4 — Noise Handling

```text
Word classes
+
<noise>
+
confidence threshold
```

Deliverable:

```text
Reduced false activations
```

---

## Milestone 5 — Robust Training

Add:

```text
Noise augmentation
Time shifting
SpecAugment
Gain variation
```

Deliverable:

```text
More robust model
```

---

## Milestone 6 — Checkpointing

Implement:

```text
best.pt
last.pt
resume
configuration saving
```

Deliverable:

```text
Recoverable cloud training
```

---

## Milestone 7 — Evaluation

Generate:

```text
Accuracy
Confusion matrix
Per-class metrics
Noise rejection
False activation rate
CPU latency
Memory usage
```

Deliverable:

```text
Complete v1 benchmark
```

---

## Milestone 8 — Inference

Build:

```text
Audio
 ↓
Prediction
 ↓
Confidence
 ↓
Noise / uncertain
 ↓
Acoustic report
```

Deliverable:

```text
Usable TinyASR v1 inference pipeline
```

---

# 40. v1 → v2 → v3 Roadmap

## v1 — Foundation

```text
                 TinyASR v1
                     │
       ┌─────────────┼─────────────┐
       │             │             │
    Log-Mel       Basic         Noise
    Features     Prosody       Handling
       │             │             │
       └─────────────┼─────────────┘
                     │
               Lightweight
                  Encoder
                     │
                     ▼
              Word Classifier
                     │
              ┌──────┴──────┐
              ▼             ▼
            Word        Uncertain
```

Focus:

> **Build a reliable baseline.**

---

# 41. v2 — Better Recognition

v2 can build directly on the v1 architecture.

Potential additions:

```text
v1 Encoder
     │
     ├───────────────┐
     ▼               ▼
   CTC Head     Attention Pool
     │               │
     └───────┬───────┘
             ▼
          Fusion
             │
             ▼
         Prediction
```

v2 can introduce:

* CTC
* Attention pooling
* Multi-head architecture
* CTC + classifier fusion
* Better prosody integration
* EMA
* Label smoothing
* Improved augmentation
* Better decoding
* More systematic uncertainty handling

Focus:

> **Improve recognition quality and robustness.**

---

# 42. v3 — Calibrated and Advanced

v3 can introduce:

```text
Advanced Prosody
       │
       ▼
Better Encoder
       │
 ┌─────┴─────┐
 ▼           ▼
CTC       Classifier
 │           │
 └─────┬─────┘
       ▼
     Fusion
       │
       ▼
 Calibration
       │
       ▼
 Abstention
```

Potential v3 features:

* Temperature scaling
* Validation-selected confidence thresholds
* Target-precision abstention
* More advanced prosody fusion
* Better uncertainty estimation
* Extensive robustness testing
* Improved deployment optimization

Focus:

> **Make the system more reliable about knowing when it does not have enough evidence.**

---

# 43. Future Emotion/Tone Architecture

The acoustic descriptors developed in v1 can eventually feed a separate analysis system.

Potential future architecture:

```text
                    Audio
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
     Speech Encoder        Acoustic Frontend
          │                       │
          │               ┌───────┴────────┐
          │               ▼                ▼
          │             Pitch           Energy
          │               │                │
          │               ▼                ▼
          │           Prosody Features
          │                   │
          └───────────┬───────┘
                      ▼
              Analysis Heads
                │      │
                ▼      ▼
             Emotion  Intent
```

This should only be developed after obtaining appropriate labeled datasets.

---

# 44. v1 Success Criteria

TinyASR v1 should be considered successful when it can:

### Recognition

* Correctly recognize the target vocabulary.
* Generalize to speakers not present in training.
* Produce repeatable predictions.

### Noise

* Recognize `<noise>` examples.
* Reject sufficiently weak predictions.
* Demonstrate measurable false-activation performance.

### Performance

* Train successfully on CPU.
* Take advantage of GPU acceleration when available.
* Resume after interruption.
* Use cached features efficiently.
* Provide measurable CPU inference latency.

### Analysis

* Extract pitch.
* Extract energy.
* Estimate voicing.
* Calculate duration.
* Produce acoustic descriptors.

### Engineering

* Save complete checkpoints.
* Reproduce experiments from configuration.
* Separate preprocessing, training, evaluation, and inference.
* Provide a clean path to v2.

---

# 45. Final v1 Definition

**TinyASR v1 is not intended to be the final architecture.**

It is the **foundation on which v2 and v3 can be measured**.

The core philosophy is:

```text
Keep v1 simple
      │
      ▼
Measure everything
      │
      ▼
Find weaknesses
      │
      ▼
Improve in v2
      │
      ▼
Calibrate and optimize in v3
```

The most important requirement is that every major improvement in a later version should be measurable against the v1 baseline.

For example:

```text
                    v1 baseline
                         │
             ┌───────────┴───────────┐
             ▼                       ▼
       USE_PROSODY=False       USE_PROSODY=True
             │                       │
             └───────────┬───────────┘
                         ▼
                   Compare Results
```

This prevents features from being added merely because they sound useful.

---

# Final Architecture

```text
                         TinyASR v1
                             │
                             ▼
                       Input Audio
                             │
                             ▼
                    Audio Preprocessing
                             │
              ┌──────────────┴──────────────┐
              │                             │
              ▼                             ▼
        Log-Mel Features             Acoustic Features
              │                             │
              │                    ┌────────┼────────┐
              │                    │        │        │
              │                    ▼        ▼        ▼
              │                  Pitch   Energy   Voicing
              │                    │        │        │
              │                    └────────┼────────┘
              │                             │
              ▼                             ▼
       Lightweight Encoder          Acoustic Descriptors
              │                             │
              ▼                             │
       Classification Head                 │
              │                             │
              ▼                             │
         Probabilities                      │
              │                             │
       ┌──────┴────────┐                    │
       │               │                    │
       ▼               ▼                    │
  High Confidence   Low Confidence          │
       │               │                    │
       ▼               ▼                    │
      Word         "uncertain"               │
       │                                    │
       └────────────────┬───────────────────┘
                        ▼
                 Final v1 Report
```

**TinyASR v1 = lightweight word recognition + noise rejection + basic uncertainty + acoustic/prosody measurement + efficient training infrastructure.**

v2 and v3 then build on this foundation rather than making v1 unnecessarily complex.


## O. Setup & config

In [ ]:
import copy
import glob
import json
import math
import os
import random
import shutil
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from IPython.display import Audio, display
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# === Environment detection (Colab vs Kaggle vs local) ===
IS_KAGGLE = os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = False

if not IS_KAGGLE:
    try:
        from google.colab import drive
        if os.path.exists("/var/colab/hostname") or any(k.startswith("COLAB_") for k in os.environ):
            IS_COLAB = True
    except ImportError:
        pass

if IS_COLAB:
    drive.mount('/content/drive', force_remount=False)
    WORK_DIR = '/content/drive/MyDrive/AI_Models/TinyASR'
elif IS_KAGGLE:
    WORK_DIR = "/kaggle/working/tinyasr_work"
else:
    WORK_DIR = "./tinyasr_work"

DATASET_ROOT = "./data/"
CACHE_LOCAL = "./cache_v3"                            # fast local disk
BACKUP_CACHE_TO_WORK_DIR = True

os.makedirs(DATASET_ROOT, exist_ok=True)      # ← add this
os.makedirs(CACHE_LOCAL, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

BEST_PATH   = os.path.join(WORK_DIR, "best.pt")
LAST_PATH   = os.path.join(WORK_DIR, "last.pt")
CAL_PATH    = os.path.join(WORK_DIR, "calibration.json")
CONFIG_PATH = os.path.join(WORK_DIR, "config.json")

print(f"Running on: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"WORK_DIR   = {WORK_DIR}")

# -- AUDIO / FEATURES ---
SAMPLE_RATE = 16_000
CLIP_SAMPLES = SAMPLE_RATE
N_FFT = 400
HOP_LENGTH = 160
N_MELS = 64

# -- VOCABULARY (CTC BLANK = 0) ---
CHARACTERS = "abcdefghijklmnopqrstuvwxyz "
BLANK_IDX = 0
CHAR_TO_INDEX = {c: i + 1 for i, c in enumerate(CHARACTERS)}
INDEX_TO_CHAR = {i: c for c, i in CHAR_TO_INDEX.items()}
INDEX_TO_CHAR[BLANK_IDX] = "_"
VOCAB_SIZE = len(CHARACTERS) + 1
NOISE_TOKEN = "<noise>"

# -- MODEL / TRAINING ---
USE_PROSODY = True
MAX_LR = 2e-3
WEIGHT_DECAY = 1e-2
EMA_DECAY = 0.999
CLS_WEIGHT = 1.0
LABEL_SMOOTHING = 0.1
PROSODY_DROPOUT = 0.2
NOISE_FRACTION = 0.05
EVAL_SUBSET = 2048
TARGET_PRECISION = 0.98
NUM_EPOCHS = None
RESUME = True

PROFILES = {
    "gpu": dict(channels=(32, 64, 96), rnn_hidden=192, rnn_layers=2, dropout=0.2,
                batch_size=256, max_epochs=60, budget_min=45),
    "cpu": dict(channels=(16, 32, 64), rnn_hidden=128, rnn_layers=1, dropout=0.1,
                batch_size=128, max_epochs=40, budget_min=90)
}
NUM_WORKERS = min(2, os.cpu_count() or 1)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
PROFILE = "gpu" if device.type == "cuda" else "cpu"
CFG = PROFILES[PROFILE]

print(f"Device: {device} | profile: {PROFILE} | mixed precision: {USE_AMP}")
print(f"Checkpoints will be saved to: {WORK_DIR}")
if PROFILE == "cpu":
    print("CPU profile: lighter model + time-budgeted training.")

## 1. Text helpers & greedy CTC decoder

In [ ]:
def text_to_tensor(text: str) -> torch.Tensor:
    indices = [CHAR_TO_INDEX[c] for c in text.lower() if c in CHAR_TO_INDEX]
    return torch.tensor(indices, dtype=torch.long)


def indices_to_text(indices) -> str:
    return "".join(INDEX_TO_CHAR[int(i)] for i in indices if int(i) != BLANK_IDX)


@torch.no_grad()
def greedy_ctc_decode(logits: torch.Tensor) -> list:
    """[Time, Batch, Vocab] -> one string per batch item (best class per frame, merge repeats, drop blanks)."""
    decoded = []
    for path in logits.argmax(dim=-1).T.tolist():
        chars, prev = [], BLANK_IDX
        for idx in path:
            if idx != prev and idx != BLANK_IDX:
                chars.append(INDEX_TO_CHAR[idx])
            prev = idx
        decoded.append("".join(chars))
    return decoded


def fix_length(waveform: torch.Tensor, num_samples: int = CLIP_SAMPLES) -> torch.Tensor:
    """Zero-pad or crop the last dimension to exactly `num_samples`."""
    n = waveform.shape[-1]
    if n < num_samples:
        return F.pad(waveform, (0, num_samples - n))
    return waveform[..., :num_samples]

## 2. Frontend: log-Mel + prosody + tone descriptors

One FP32 module produces everything from the raw waveform:

* **70 feature channels per 10 ms frame** = 64 log-Mel bins + 6 prosody channels
  (`energy`, `log_f0`, `voiced`, `pitch_strength`, `zcr`, `centroid`), each normalised per utterance.
* **8 absolute descriptors per clip** (`STAT_NAMES`) – loudness in dBFS, mean pitch in Hz, pitch slope in
  semitones/s, … – used for the tone report and the "no speech" gate.

Pitch is tracked with a windowed normalised autocorrelation (40 ms window, 60–400 Hz search range,
parabolic peak refinement and a small octave-error penalty).

In [ ]:
WIN_P = 640                                            # 40 ms analysis window for pitch / energy
LAG_MIN = SAMPLE_RATE // 400                           # 400 Hz upper pitch limit
LAG_MAX = SAMPLE_RATE // 60                            # 60 Hz lower pitch limit
OCTAVE_COST = 0.05                                     # prefers the shorter lag when two peaks are nearly equal
VOICED_THR = 0.45                                      # min normalised autocorrelation to call a frame voiced
ACTIVE_DB = 35.0                                       # frame is "active" if within 35 dB of the loudest frame ...
ABS_FLOOR_DB = -65.0                                   # ... and louder than this absolute floor

N_PROS = 6
PROSODY_NAMES = ["energy", "log_f0", "voiced", "pitch_strength", "zcr", "centroid"]
STAT_NAMES = ["loudness_db", "peak_db", "active_ratio", "voiced_ratio",
              "f0_mean_hz", "f0_std_st", "f0_slope_st_s", "centroid_hz"]
N_FEAT = N_MELS + N_PROS


def _mmean(v, m):
    return (v * m).sum(-1) / m.sum(-1).clamp_min(1.0)


def _mstd(v, m):
    mu = _mmean(v, m).unsqueeze(-1)
    return ((((v - mu) ** 2) * m).sum(-1) / m.sum(-1).clamp_min(1.0)).sqrt()


class Frontend(nn.Module):
    """waveform [B, S] -> (feats [B, 70, T] normalised, stats [B, 8] absolute). Always FP32."""

    def __init__(self):
        super().__init__()
        self.mel = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT,
                                    hop_length=HOP_LENGTH, n_mels=N_MELS)
        self.to_db = T.AmplitudeToDB(stype="power", top_db=80.0)
        win = torch.hann_window(WIN_P)
        win_ac = torch.fft.irfft(torch.fft.rfft(win, n=2 * WIN_P).abs() ** 2, n=2 * WIN_P)[: LAG_MAX + 2]
        self.register_buffer("win", win)
        self.register_buffer("win_ac", win_ac.clamp_min(1e-8))          # window autocorrelation (bias correction)
        self.register_buffer("freqs", torch.arange(WIN_P + 1) * (SAMPLE_RATE / (2 * WIN_P)))
        self.register_buffer("lags", torch.arange(LAG_MIN, LAG_MAX + 1, dtype=torch.float32))

    @torch.no_grad()
    def raw(self, wave: torch.Tensor) -> dict:
        """Per-frame measurements in physical units (used for plots and for building the features)."""
        wave = wave.float()
        with torch.autocast(device_type=wave.device.type, enabled=False):
            mel_db = self.to_db(self.mel(wave.unsqueeze(1)))[:, 0]           # [B, n_mels, T]  (4-D call -> per-clip top_db)
            pad = WIN_P // 2
            padded = F.pad(wave.unsqueeze(1), (pad, pad), mode="reflect").squeeze(1)
            frames = padded.unfold(-1, WIN_P, HOP_LENGTH)                    # [B, T, WIN_P]  (T matches the Mel frames)

            rms = frames.pow(2).mean(-1).clamp_min(1e-10).sqrt()
            energy_db = 20 * torch.log10(rms)                                # volume per frame, dBFS
            zcr = ((frames[..., 1:] * frames[..., :-1]) < 0).float().mean(-1)

            spec = torch.fft.rfft(frames * self.win, n=2 * WIN_P)
            power = spec.real ** 2 + spec.imag ** 2
            centroid = (power * self.freqs).sum(-1) / power.sum(-1).clamp_min(1e-10)    # brightness, Hz

            ac = torch.fft.irfft(power, n=2 * WIN_P)[..., : LAG_MAX + 2]
            nccf = ac / self.win_ac
            nccf = nccf / nccf[..., :1].clamp_min(1e-10)
            seg = nccf[..., LAG_MIN - 1: LAG_MAX + 2]
            core = seg[..., 1:-1]
            score = core - OCTAVE_COST * torch.log2(self.lags / LAG_MIN)
            idx = score.argmax(-1, keepdim=True)
            strength = core.gather(-1, idx).squeeze(-1)
            y0, y1, y2 = seg.gather(-1, idx), seg.gather(-1, idx + 1), seg.gather(-1, idx + 2)
            den = y0 - 2 * y1 + y2
            delta = torch.where(den.abs() > 1e-8, 0.5 * (y0 - y2) / den, torch.zeros_like(den)).clamp(-1, 1).squeeze(-1)
            f0 = SAMPLE_RATE / (self.lags[idx.squeeze(-1)] + delta)

            active = (energy_db > energy_db.amax(-1, keepdim=True) - ACTIVE_DB) & (energy_db > ABS_FLOOR_DB)
            voiced = active & (strength > VOICED_THR)
            f0 = torch.where(voiced, f0, torch.zeros_like(f0))
        return dict(mel_db=mel_db, energy_db=energy_db, zcr=zcr, centroid=centroid, strength=strength,
                    f0=f0, active=active, voiced=voiced, peak=wave.abs().amax(-1))

    @torch.no_grad()
    def forward(self, wave: torch.Tensor):
        r = self.raw(wave)

        def z(v):
            return (v - v.mean(-1, keepdim=True)) / (v.std(-1, keepdim=True) + 1e-5)

        vm, am = r["voiced"].float(), r["active"].float()
        log_f0 = torch.log(r["f0"].clamp_min(1.0))
        mu = _mmean(log_f0, vm).unsqueeze(-1)
        sd = _mstd(log_f0, vm).unsqueeze(-1).clamp_min(0.03)                 # floor ~0.5 semitone
        prosody = torch.stack([
            z(r["energy_db"].clamp_min(-80)),                                # energy
            (log_f0 - mu) / sd * vm,                                         # log_f0 (speaker-normalised, 0 if unvoiced)
            vm - 0.5,                                                        # voiced
            r["strength"].clamp(0, 1) - 0.5,                                 # pitch_strength
            z(r["zcr"]),                                                     # zcr
            z(r["centroid"]),                                                # centroid
        ], dim=1)
        mel = r["mel_db"]
        mel_z = (mel - mel.mean((-2, -1), keepdim=True)) / (mel.std((-2, -1), keepdim=True) + 1e-5)
        feats = torch.cat([mel_z, prosody], dim=1)                           # [B, 70, T]

        # ── absolute, human-readable descriptors ──
        e = r["energy_db"]
        n_act, n_voi = am.sum(-1), vm.sum(-1)
        loudness = torch.where(n_act > 0, _mmean(e, am), e.mean(-1))
        peak = 20 * torch.log10(r["peak"].clamp_min(1e-5))
        centroid_hz = torch.where(n_act > 0, _mmean(r["centroid"], am), r["centroid"].mean(-1))
        semis = (12.0 / math.log(2)) * log_f0
        f0_mean = torch.where(n_voi > 0, torch.exp(_mmean(log_f0, vm)), torch.zeros_like(loudness))
        f0_std = torch.where(n_voi > 1, _mstd(semis, vm), torch.zeros_like(loudness))
        t = (torch.arange(e.size(-1), device=e.device, dtype=e.dtype) * (HOP_LENGTH / SAMPLE_RATE)).expand_as(e)
        tm, ym = _mmean(t, vm).unsqueeze(-1), _mmean(semis, vm).unsqueeze(-1)
        cov = ((t - tm) * (semis - ym) * vm).sum(-1) / n_voi.clamp_min(1.0)
        var_t = ((t - tm) ** 2 * vm).sum(-1) / n_voi.clamp_min(1.0)
        slope = torch.where(n_voi > 2, cov / var_t.clamp_min(1e-6), torch.zeros_like(cov))   # semitones / second
        stats = torch.stack([loudness, peak, am.mean(-1), n_voi / n_act.clamp_min(1.0),
                             f0_mean, f0_std, slope, centroid_hz], dim=-1)
        return feats, stats


frontend = Frontend().to(device)

### 2b. Self-check of the pitch / volume tracker (synthetic voices with a known answer)

In [ ]:
def harmonic_tone(f0, gain=0.3, glide_to=None, dur=1.0):
    n = int(dur * SAMPLE_RATE)
    t = torch.arange(n) / SAMPLE_RATE
    f = f0 if glide_to is None else f0 + (glide_to - f0) * t / dur
    phase = 2 * math.pi * torch.cumsum(f / SAMPLE_RATE * torch.ones(n), 0)
    w = sum(torch.sin(h * phase) / h for h in range(1, 9))
    return w / w.abs().max() * gain


cases = {"110 Hz": harmonic_tone(110), "180 Hz": harmonic_tone(180), "250 Hz": harmonic_tone(250),
         "330 Hz": harmonic_tone(330), "glide 150->250": harmonic_tone(150, glide_to=250),
         "glide 250->150": harmonic_tone(250, glide_to=150), "180 Hz, -20 dB": harmonic_tone(180, 0.03),
         "white noise": 0.2 * torch.randn(SAMPLE_RATE), "silence": torch.zeros(SAMPLE_RATE)}
_, st = frontend(torch.stack(list(cases.values())).to(device))
st = st.cpu()
print(f"{'case':16s}" + "".join(f"{n:>15s}" for n in STAT_NAMES))
for name, row in zip(cases, st):
    print(f"{name:16s}" + "".join(f"{v:15.2f}" for v in row.tolist()))

S = {k: st[i] for i, k in enumerate(cases)}
for name, hz in [("110 Hz", 110), ("180 Hz", 180), ("250 Hz", 250), ("330 Hz", 330)]:
    assert abs(S[name][4] / hz - 1) < 0.03, f"pitch tracker off for {name}: {S[name][4]:.1f} Hz"
assert S["glide 150->250"][6] > 5 and S["glide 250->150"][6] < -5, "pitch slope sign wrong"
assert 18 < S["180 Hz"][0] - S["180 Hz, -20 dB"][0] < 22, "loudness should drop ~20 dB"
assert S["white noise"][3] < 0.2 and S["silence"][0] < -90, "noise/silence should not look like voice"
print("\n✓ pitch, volume, contour and voicing checks passed")

## 3. Feature cache (runs once)
Speech Commands clips **and** synthetic-or-real background-noise clips (`<noise>` class) are converted once into
FP16 features (`[N, 70, 101]`) + FP32 descriptors (`[N, 8]`). Cache folders are tagged with the feature settings, so
changing `N_MELS`/`HOP_LENGTH` never reuses a stale cache. A copy is stored in `WORK_DIR`, which survives session resets.

In [ ]:
CACHE_TAG = f"m{N_MELS}_f{N_FFT}_h{HOP_LENGTH}_p{N_PROS}"
CACHE_DIR = os.path.join(CACHE_LOCAL, CACHE_TAG)
BACKUP_DIR = os.path.join(WORK_DIR, "cache_v3", CACHE_TAG)
os.makedirs(CACHE_DIR, exist_ok=True)


def cache_files(name, root):
    return {k: os.path.join(root, f"{name}_{k}") for k in ("feats.npy", "stats.npy", "labels.json")}


def cache_complete(name, root):
    return all(os.path.exists(p) for p in cache_files(name, root).values())     # labels.json is written LAST


def copy_cache(name, src_root, dst_root):
    os.makedirs(dst_root, exist_ok=True)
    for k, src in cache_files(name, src_root).items():
        shutil.copyfile(src, cache_files(name, dst_root)[k])


@torch.no_grad()
def build_cache(name, n, batches):
    """`batches` yields (waveforms [B, S], labels list[str]). Writes through a memmap, so RAM never spikes."""
    if cache_complete(name, CACHE_DIR):
        print(f"[{name}] cache found - skipping")
        return
    if BACKUP_CACHE_TO_WORK_DIR and os.path.abspath(BACKUP_DIR) != os.path.abspath(CACHE_DIR) and cache_complete(name, BACKUP_DIR):
        print(f"[{name}] restoring cache from {BACKUP_DIR}")
        copy_cache(name, BACKUP_DIR, CACHE_DIR)
        return

    paths = cache_files(name, CACHE_DIR)
    frames = frontend(torch.zeros(1, CLIP_SAMPLES, device=device))[0].shape[-1]
    feats = np.lib.format.open_memmap(paths["feats.npy"], mode="w+", dtype=np.float16, shape=(n, N_FEAT, frames))
    stats_all, labels, i = [], [], 0
    for waveforms, batch_labels in tqdm(batches, desc=f"caching {name}", total=math.ceil(n / 256)):
        f, s = frontend(waveforms.to(device))
        feats[i:i + len(batch_labels)] = f.half().cpu().numpy()
        stats_all.append(s.cpu().numpy())
        labels.extend(batch_labels)
        i += len(batch_labels)
    assert i == n, f"[{name}] expected {n} clips, cached {i}"
    feats.flush()
    del feats
    np.save(paths["stats.npy"], np.concatenate(stats_all).astype(np.float32))
    with open(paths["labels.json"], "w") as f:
        json.dump(labels, f)
    if BACKUP_CACHE_TO_WORK_DIR and os.path.abspath(BACKUP_DIR) != os.path.abspath(CACHE_DIR):
        copy_cache(name, CACHE_DIR, BACKUP_DIR)


# ── speech ───────────────────────────────────────────────────────
def load_split(subset: str):
    return torchaudio.datasets.SPEECHCOMMANDS(
        root=DATASET_ROOT, url="speech_commands_v0.02", download=True, subset=subset)


def raw_collate(batch):
    return torch.stack([fix_length(item[0])[0] for item in batch]), [item[2] for item in batch]


def speech_batches(dataset):
    return DataLoader(dataset, batch_size=256, shuffle=False, collate_fn=raw_collate, num_workers=NUM_WORKERS)


# ── noise / silence ──────────────────────────────────────────────
def synthetic_noise_bank(seconds=60, seed=0):
    g = torch.Generator().manual_seed(seed)
    n = seconds * SAMPLE_RATE
    white = torch.randn(n, generator=g)
    spec = torch.fft.rfft(white)
    f = torch.arange(spec.numel()).clamp_min(1).float()
    pink = torch.fft.irfft(spec / f.sqrt(), n=n)
    brown = torch.fft.irfft(spec / f, n=n)
    t = torch.arange(n) / SAMPLE_RATE
    hum = torch.sin(2 * math.pi * 50 * t) + 0.5 * torch.sin(2 * math.pi * 100 * t) + 0.05 * white
    return [w / w.abs().max() for w in (white, pink, brown, hum)]


def load_noise_bank():
    folder = os.path.join(DATASET_ROOT, "SpeechCommands", "speech_commands_v0.02", "_background_noise_")
    bank = []
    for path in sorted(glob.glob(os.path.join(folder, "*.wav"))):
        try:
            w, sr = torchaudio.load(path)
            w = w.mean(0)
            bank.append(torchaudio.functional.resample(w, sr, SAMPLE_RATE) if sr != SAMPLE_RATE else w)
        except Exception as e:                                         # e.g. missing audio backend
            print(f"  could not read {os.path.basename(path)}: {e}")
    if bank:
        print(f"Noise bank: {len(bank)} real background-noise files")
        return bank
    print("Noise bank: no real files readable -> using synthetic white/pink/brown/hum noise")
    return synthetic_noise_bank()


def noise_batches(bank, n, portion, seed, silence_share=0.15, batch=256):
    """`portion` = which part of every file may be used (train/val/test never overlap in time)."""
    g = torch.Generator().manual_seed(seed)
    for s in range(0, n, batch):
        b = min(batch, n - s)
        clips = []
        for _ in range(b):
            if torch.rand(1, generator=g).item() < silence_share:            # digital-ish silence + faint hiss
                clip = 1e-4 * torch.rand(1, generator=g).item() * torch.randn(CLIP_SAMPLES, generator=g)
            else:
                src = bank[int(torch.randint(len(bank), (1,), generator=g))]
                lo = int(len(src) * portion[0])
                hi = max(int(len(src) * portion[1]) - CLIP_SAMPLES, lo + 1)
                start = int(torch.randint(lo, hi, (1,), generator=g))
                clip = fix_length(src[start:start + CLIP_SAMPLES])
                clip = clip / clip.abs().max().clamp_min(1e-8)
                clip = clip * 10 ** ((-40 + 34 * torch.rand(1, generator=g).item()) / 20)   # peak -40 ... -6 dBFS
            clips.append(clip)
        yield torch.stack(clips), [NOISE_TOKEN] * b


noise_bank = load_noise_bank()
NOISE_PORTIONS = {"train": (0.0, 0.7), "val": (0.7, 0.85), "test": (0.85, 1.0)}

for split_name, subset in [("train", "training"), ("val", "validation"), ("test", "testing")]:
    ds = load_split(subset)
    build_cache(split_name, len(ds), speech_batches(ds))
    n_noise = max(int(NOISE_FRACTION * len(ds)), 64)
    build_cache(f"{split_name}_noise", n_noise,
                noise_batches(noise_bank, n_noise, NOISE_PORTIONS[split_name], seed={"train": 1, "val": 2, "test": 3}[split_name]))
print("Cache ready in", CACHE_DIR)

## 4. Load splits onto the device
Whole splits live in memory (GPU if there is one) as FP16 – training needs no DataLoader and almost no CPU.

In [ ]:
def load_features_to_device(feat_paths, chunk=8192):
    arrays = [np.load(p, mmap_mode="r") for p in feat_paths]
    total = sum(len(a) for a in arrays)
    feats = torch.empty((total, *arrays[0].shape[1:]), dtype=torch.float16, device=device)
    pos = 0
    for a in arrays:
        for s in range(0, len(a), chunk):                              # chunked copy -> no RAM spike
            block = torch.from_numpy(np.array(a[s:s + chunk])).to(device)
            feats[pos:pos + len(block)] = block
            pos += len(block)
    return feats


def set_lexicon(words):
    """Defines the closed vocabulary (+ <noise>) and the CTC candidate targets used for lexicon scoring."""
    global LEXICON, WORD_TO_ID, NOISE_ID, cand_targets, cand_lengths
    LEXICON = list(words) + [NOISE_TOKEN]
    WORD_TO_ID = {w: i for i, w in enumerate(LEXICON)}
    NOISE_ID = WORD_TO_ID[NOISE_TOKEN]
    texts = [text_to_tensor("" if w == NOISE_TOKEN else w) for w in LEXICON]
    cand_targets = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=BLANK_IDX).to(device)
    cand_lengths = torch.tensor([len(t) for t in texts])                # CPU (CTC wants CPU lengths)


class Split:
    """One dataset split (speech + its noise clips), fully resident on the device."""

    def __init__(self, names):
        files = [cache_files(n, CACHE_DIR) for n in names]
        self.feats = load_features_to_device([f["feats.npy"] for f in files])            # [N, 70, T] FP16
        self.stats = torch.from_numpy(np.concatenate([np.load(f["stats.npy"]) for f in files]))   # CPU [N, 8]
        labels = []
        for f in files:
            with open(f["labels.json"]) as fh:
                labels += json.load(fh)
        self.n = len(labels)
        assert self.n == len(self.feats) == len(self.stats)
        self.word_ids = torch.tensor([WORD_TO_ID[w] for w in labels])                     # CPU
        encoded = [text_to_tensor("" if w == NOISE_TOKEN else w) for w in labels]
        self.target_lengths = torch.tensor([len(t) for t in encoded])                      # CPU
        self.targets = nn.utils.rnn.pad_sequence(encoded, batch_first=True, padding_value=BLANK_IDX).to(device)

    def order(self, shuffle=False, limit=None, seed=None):
        if shuffle:
            o = torch.randperm(self.n)
        elif seed is not None:                                          # fixed RANDOM subset (files are sorted by word!)
            o = torch.randperm(self.n, generator=torch.Generator().manual_seed(seed))
        else:
            o = torch.arange(self.n)
        return o[:limit] if limit else o

    def batches(self, batch_size, order):
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            idx_dev = idx.to(device)
            yield (self.feats[idx_dev].float(), self.targets[idx_dev],
                   self.target_lengths[idx], self.word_ids[idx])


with open(cache_files("train", CACHE_DIR)["labels.json"]) as f:
    set_lexicon(sorted(set(json.load(f))))                               # the closed vocabulary (35 words) + <noise>
print(f"{len(LEXICON) - 1} words + {NOISE_TOKEN}:", ", ".join(LEXICON))

train_split = Split(["train", "train_noise"])
val_split = Split(["val", "val_noise"])
test_split = Split(["test", "test_noise"])
print(f"train {train_split.n:,} | val {val_split.n:,} | test {test_split.n:,} | "
      f"features (train): {tuple(train_split.feats.shape)}")

## 5. Model, augmentation & loss
* `Net` = 3 conv blocks over the Mel image → (+ prosody channels, pooled to the same frame rate) → BiGRU →
  **CTC head** (per-frame characters) and **classifier head** (attention-pooled, one logit per lexicon word).
* Augmentation (all on the accelerator): time-shift, SpecAugment on the Mel rows, and *prosody dropout*
  so the network can't over-rely on pitch/energy (which are also less reliable on noisy audio).

In [ ]:
class Net(nn.Module):
    """x [B, 70, T] -> (ctc_logits [T/2, B, vocab], word_logits [B, n_classes])."""

    def __init__(self, n_classes, use_prosody=True, channels=(32, 64, 96), rnn_hidden=192,
                 rnn_layers=2, dropout=0.2):
        super().__init__()
        self.use_prosody = use_prosody

        def block(c_in, c_out, pool):
            return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
                    nn.BatchNorm2d(c_out), nn.ReLU(inplace=True), nn.MaxPool2d(pool)]

        c1, c2, c3 = channels
        self.cnn = nn.Sequential(*block(1, c1, (2, 2)),          # mel /2, time /2
                                 *block(c1, c2, (2, 1)),         # mel /2
                                 *block(c2, c3, (2, 1)))         # mel /2  -> n_mels / 8
        feat_dim = c3 * (N_MELS // 8) + (N_PROS if use_prosody else 0)
        self.rnn = nn.GRU(feat_dim, rnn_hidden, num_layers=rnn_layers, batch_first=True,
                          bidirectional=True, dropout=dropout if rnn_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.ctc_head = nn.Linear(2 * rnn_hidden, VOCAB_SIZE)
        self.attn = nn.Linear(2 * rnn_hidden, 1)
        self.cls_head = nn.Linear(2 * rnn_hidden, n_classes)

    def forward(self, x):
        mel = x[:, :N_MELS].unsqueeze(1)                          # [B, 1, n_mels, T]
        h = self.cnn(mel)                                         # [B, C, n_mels/8, T/2]
        b, c, m, t = h.shape
        h = h.reshape(b, c * m, t)
        if self.use_prosody:
            h = torch.cat([h, F.avg_pool1d(x[:, N_MELS:], 2, 2)], dim=1)    # prosody at the same frame rate
        h, _ = self.rnn(h.permute(0, 2, 1))                       # [B, T/2, 2H]
        h = self.dropout(h)
        ctc_logits = self.ctc_head(h).permute(1, 0, 2)            # [T/2, B, vocab]
        w = torch.softmax(self.attn(h).squeeze(-1), dim=1)        # attention over time
        word_logits = self.cls_head((w.unsqueeze(-1) * h).sum(1))
        return ctc_logits, word_logits


def build_model():
    return Net(len(LEXICON), use_prosody=USE_PROSODY, channels=CFG["channels"], rnn_hidden=CFG["rnn_hidden"],
               rnn_layers=CFG["rnn_layers"], dropout=CFG["dropout"]).to(device)


# ── Augmentation ─────────────────────────────────────────────────
def time_shift(x, max_shift=10):
    """Shift each clip by up to ±max_shift frames (10 ms each), zero-filled. x: [B, C, T]."""
    b, c, t = x.shape
    shift = torch.randint(-max_shift, max_shift + 1, (b,), device=x.device)
    src = torch.arange(t, device=x.device)[None, :] - shift[:, None]
    valid = (src >= 0) & (src < t)
    return torch.gather(x, 2, src.clamp(0, t - 1)[:, None, :].expand(b, c, t)) * valid[:, None, :]


def freq_mask(x, max_width=8):
    b, c, _ = x.shape
    width = torch.randint(0, max_width + 1, (b,), device=x.device)
    start = (torch.rand(b, device=x.device) * (N_MELS - width)).long()
    pos = torch.arange(c, device=x.device)[None, :]
    mask = (pos >= start[:, None]) & (pos < (start + width)[:, None]) & (pos < N_MELS)   # Mel rows only
    return x.masked_fill(mask[:, :, None], 0.0)


def time_mask(x, max_width=12):
    b, _, t = x.shape
    width = torch.randint(0, max_width + 1, (b,), device=x.device)
    start = (torch.rand(b, device=x.device) * (t - width)).long()
    pos = torch.arange(t, device=x.device)[None, :]
    mask = (pos >= start[:, None]) & (pos < (start + width)[:, None])
    return x.masked_fill(mask[:, None, :], 0.0)


def augment(x):
    x = time_shift(x)
    for _ in range(2):
        x = freq_mask(x)
    for _ in range(2):
        x = time_mask(x)
    keep = (torch.rand(x.size(0), 1, 1, device=x.device) >= PROSODY_DROPOUT).float()
    return torch.cat([x[:, :N_MELS], x[:, N_MELS:] * keep], dim=1)


# ── Loss ─────────────────────────────────────────────────────────
criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)


def ctc_loss(logits, targets, target_lengths):
    """Raw logits [T, B, V] -> scalar. log_softmax + CTC always in FP32 (CUDA CTC has no FP16 kernel)."""
    log_probs = F.log_softmax(logits.float(), dim=-1)
    input_lengths = torch.full((logits.size(1),), logits.size(0), dtype=torch.long)
    return criterion(log_probs, targets, input_lengths, target_lengths)


def total_loss(ctc_logits, word_logits, targets, target_lengths, word_ids):
    ce = F.cross_entropy(word_logits.float(), word_ids.to(word_logits.device), label_smoothing=LABEL_SMOOTHING)
    return ctc_loss(ctc_logits, targets, target_lengths) + CLS_WEIGHT * ce


print(f"Parameters ({PROFILE} profile, prosody={'on' if USE_PROSODY else 'off'}): "
      f"{sum(p.numel() for p in build_model().parameters()):,}")

## 6. Decoding, calibration & abstention

For every clip we get two views of "which word is it":
* `ll` – the CTC log-likelihood of each lexicon word (`-CTC loss`), and
* `cls` – the classifier's log-probabilities.

They are fused (`alpha * ll + (1-alpha) * cls`), a **temperature** turns the fused scores into honest
probabilities, and a **confidence threshold** (chosen on validation data for `TARGET_PRECISION`) decides
whether to answer at all. Everything below is fitted on the validation set and reported on the untouched test set.

In [ ]:
@torch.no_grad()
def lexicon_ll(ctc_logits):
    """ctc_logits [T, B, V] -> CTC log-likelihood of every lexicon word: [B, W] (higher = better)."""
    t, b, v = ctc_logits.shape
    w = cand_targets.size(0)
    lp = F.log_softmax(ctc_logits.float(), dim=-1).unsqueeze(2).expand(t, b, w, v).reshape(t, b * w, v)
    tg = cand_targets.unsqueeze(0).expand(b, w, -1).reshape(b * w, -1)
    tl = cand_lengths.unsqueeze(0).expand(b, w).reshape(b * w)
    il = torch.full((b * w,), t, dtype=torch.long)
    # zero_infinity must be False: it would turn "impossible" into 0 loss, i.e. the BEST score
    nll = F.ctc_loss(lp, tg, il, tl, blank=BLANK_IDX, reduction="none", zero_infinity=False)
    return -torch.nan_to_num(nll, nan=1e9, posinf=1e9).view(b, w)


@torch.no_grad()
def collect(net, split, order):
    """Runs the model over `order` and returns everything needed for metrics / calibration (on CPU)."""
    net.eval()
    ll, cls, ids, greedy, loss_sum, n_batches = [], [], [], [], 0.0, 0
    for x, targets, target_lengths, word_ids in split.batches(CFG["batch_size"], order):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            ctc_logits, word_logits = net(x)
        loss_sum += total_loss(ctc_logits, word_logits, targets, target_lengths, word_ids).item()
        n_batches += 1
        ll.append(lexicon_ll(ctc_logits).cpu())
        cls.append(F.log_softmax(word_logits.float(), dim=-1).cpu())
        ids.append(word_ids)
        greedy += greedy_ctc_decode(ctc_logits)
    return {"ll": torch.cat(ll), "cls": torch.cat(cls), "ids": torch.cat(ids),
            "greedy": greedy, "loss": loss_sum / max(n_batches, 1), "order": order}


def fuse(res, alpha):
    return alpha * res["ll"] + (1 - alpha) * res["cls"]


def quick_metrics(res, alpha=0.5):
    ids = res["ids"]
    truth = ["" if LEXICON[i] == NOISE_TOKEN else LEXICON[i] for i in ids.tolist()]
    return {"loss": res["loss"],
            "greedy": float(np.mean([g == t for g, t in zip(res["greedy"], truth)])),
            "ctc": (res["ll"].argmax(1) == ids).float().mean().item(),
            "cls": (res["cls"].argmax(1) == ids).float().mean().item(),
            "fused": (fuse(res, alpha).argmax(1) == ids).float().mean().item()}


def fit_temperature(logits, ids):
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), ids)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def fit_confidence_threshold(conf, correct, target):
    """Largest set of most-confident answers whose accuracy is still >= target -> its lowest confidence."""
    order = conf.argsort(descending=True)
    c = correct[order].float()
    running = c.cumsum(0) / torch.arange(1, len(c) + 1)
    ok = (running >= target).nonzero()
    if len(ok) == 0:
        print(f"  ! target precision {target:.0%} is not reachable on validation data; answering only the single most confident clip")
        return float(conf[order][0])
    return float(conf[order][int(ok.max())])


def fit_gate(stats_speech):
    """'No speech' gate from the training speech distribution (loudness + active fraction)."""
    loud = stats_speech[:, STAT_NAMES.index("loudness_db")]
    act = stats_speech[:, STAT_NAMES.index("active_ratio")]
    return {"min_loudness_db": float(torch.quantile(loud, 0.005)) - 3.0,
            "min_active_ratio": max(0.02, float(torch.quantile(act, 0.005)) * 0.5)}


def tone_percentiles(stats_speech):
    """Reference percentiles (training speakers) to put words like 'soft' / 'high' on the raw numbers."""
    out = {}
    for name in ["loudness_db", "f0_mean_hz", "centroid_hz"]:
        v = stats_speech[:, STAT_NAMES.index(name)]
        if name == "f0_mean_hz":
            v = v[v > 0]
        out[name] = [float(torch.quantile(v, 0.2)), float(torch.quantile(v, 0.8))]
    return out


def calibrate(val_res, val_stats):
    """Pick alpha + temperature + confidence threshold on the validation set."""
    ids = val_res["ids"]
    best = None
    for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
        logits = fuse(val_res, alpha)
        temp = fit_temperature(logits, ids)
        nll = F.cross_entropy(logits / temp, ids).item()
        acc = (logits.argmax(1) == ids).float().mean().item()
        print(f"  alpha {alpha:.2f} | temperature {temp:5.2f} | val NLL {nll:.4f} | val acc {acc:.2%}")
        if best is None or nll < best[0]:
            best = (nll, alpha, temp)
    _, alpha, temp = best
    probs = torch.softmax(fuse(val_res, alpha) / temp, dim=1)
    conf, pred = probs.max(1)
    thr = fit_confidence_threshold(conf, pred == ids, TARGET_PRECISION)
    speech = val_stats[val_res["ids"] != NOISE_ID]
    cal = {"lexicon": LEXICON, "alpha": alpha, "temperature": temp, "conf_threshold": thr,
           "target_precision": TARGET_PRECISION, "gate": fit_gate(speech), "tone_ref": tone_percentiles(speech)}
    return cal

## 7. Training
* **Time budget:** a few real training steps are timed on a throw-away model, then the number of epochs is chosen to
  fit `CFG["budget_min"]` minutes (override with `NUM_EPOCHS`). One-cycle LR, AdamW, EMA weights, grad-clipping.
* **Resume:** the full state is saved every epoch to `WORK_DIR`. If the session dies, re-run the notebook: this cell
  picks up at the next epoch with the same schedule.

In [ ]:
class EMA:
    """Exponential moving average of the weights (and BatchNorm statistics), with a warm-up so early epochs aren't dominated by the random init."""

    def __init__(self, net, decay):
        self.decay, self.n = decay, 0
        self.module = copy.deepcopy(net).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self._src, self._dst = net.state_dict(), self.module.state_dict()      # tensors shared with the live models

    @torch.no_grad()
    def update(self, net):
        self.n += 1
        d = min(self.decay, (1.0 + self.n) / (10.0 + self.n))
        floats = [k for k, v in self._dst.items() if v.is_floating_point()]
        torch._foreach_lerp_([self._dst[k] for k in floats], [self._src[k] for k in floats], 1.0 - d)
        for k, v in self._dst.items():                                          # integer buffers (BN step counters)
            if not v.is_floating_point():
                v.copy_(self._src[k])

    def state_dict(self):
        return {"module": self.module.state_dict(), "n": self.n}

    def load_state_dict(self, sd):
        self.module.load_state_dict(sd["module"])
        self.n = sd["n"]


def train_step(net, opt, scaler, x, targets, target_lengths, word_ids, sched=None, ema=None):
    net.train()
    x = augment(x)
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        ctc_logits, word_logits = net(x)
    loss = total_loss(ctc_logits, word_logits, targets, target_lengths, word_ids)      # FP32, outside autocast
    if not torch.isfinite(loss):
        return None
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    scaler.step(opt)
    scaler.update()
    if sched is not None:
        sched.step()
    if ema is not None:
        ema.update(net)
    return loss.item()


def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def plan_epochs():
    """Time a few steps + one quick validation on a throw-away model, then fit the epochs into the budget."""
    bench = build_model()
    opt = torch.optim.AdamW(bench.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    it = train_split.batches(CFG["batch_size"], train_split.order(shuffle=True, limit=CFG["batch_size"] * 8))
    times = []
    for i, batch in enumerate(it):
        sync()
        t0 = time.time()
        train_step(bench, opt, scaler, *batch)
        sync()
        if i >= 2:                                                       # first steps include warm-up
            times.append(time.time() - t0)
    t_step = float(np.median(times))
    probe = val_split.order(seed=0, limit=512)
    sync()
    t0 = time.time()
    collect(bench, val_split, probe)
    sync()
    t_eval = (time.time() - t0) * (min(EVAL_SUBSET, val_split.n) / len(probe))
    steps = math.ceil(train_split.n / CFG["batch_size"])
    epoch_s = steps * t_step + t_eval
    epochs = int(max(3, min(CFG["max_epochs"], (CFG["budget_min"] * 60) // epoch_s)))
    print(f"Timing: {t_step * 1000:.0f} ms/step x {steps} steps + {t_eval:.0f}s validation = {epoch_s / 60:.1f} min/epoch")
    return epochs


steps_per_epoch = math.ceil(train_split.n / CFG["batch_size"])
SIG = {"profile": PROFILE, "use_prosody": USE_PROSODY, "n_classes": len(LEXICON),
       "batch_size": CFG["batch_size"], "n_train": train_split.n}

model = build_model()
ema = EMA(model, EMA_DECAY)
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

state = None
if RESUME and os.path.exists(LAST_PATH):
    try:
        state = torch.load(LAST_PATH, map_location=device, weights_only=False)
        if state["sig"] != SIG:
            print("Found a checkpoint from a different configuration - starting fresh.")
            state = None
    except Exception as e:
        print(f"Could not read {LAST_PATH} ({e}) - starting fresh.")
        state = None

if state is not None:
    epochs_total = state["epochs"]
    print(f"Resuming after epoch {state['epoch']} of {epochs_total}")
else:
    epochs_total = NUM_EPOCHS or plan_epochs()
    if epochs_total < 15:
        print(f"! Only {epochs_total} epochs fit the {CFG['budget_min']}-minute budget. Raise CFG['budget_min'] "
              f"(and resume across sessions) or use a GPU if you want the full-length schedule.")

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, total_steps=epochs_total * steps_per_epoch, pct_start=0.15)
start_epoch, best_score = 1, (-1.0, 0.0)
if state is not None:
    model.load_state_dict(state["model"])
    ema.load_state_dict(state["ema"])
    optimizer.load_state_dict(state["opt"])
    scheduler.load_state_dict(state["sched"])
    scaler.load_state_dict(state["scaler"])
    start_epoch, best_score = state["epoch"] + 1, tuple(state["best"])

print(f"Plan: {epochs_total} epochs x {steps_per_epoch} steps | batch {CFG['batch_size']} | "
      f"{sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
val_quick_order = val_split.order(seed=0, limit=EVAL_SUBSET)
print("=" * 110)
for epoch in range(start_epoch, epochs_total + 1):
    t0 = time.time()
    running, n_ok, n_skipped = 0.0, 0, 0
    order = train_split.order(shuffle=True)
    for batch in tqdm(train_split.batches(CFG["batch_size"], order), total=steps_per_epoch,
                      desc=f"epoch {epoch}/{epochs_total}", leave=False):
        loss = train_step(model, optimizer, scaler, *batch, sched=scheduler, ema=ema)
        if loss is None:
            n_skipped += 1
        else:
            running += loss
            n_ok += 1

    m = quick_metrics(collect(ema.module, val_split, val_quick_order))
    print(f"Epoch [{epoch:02d}/{epochs_total}] train {running / max(n_ok, 1):.3f} | val loss {m['loss']:.3f} | "
          f"acc: greedy {m['greedy']:.1%}  ctc-lexicon {m['ctc']:.1%}  classifier {m['cls']:.1%}  FUSED {m['fused']:.1%} | "
          f"lr {scheduler.get_last_lr()[0]:.1e} | skipped {n_skipped} | {time.time() - t0:.0f}s")

    score = (m["fused"], -m["loss"])
    if score > best_score:
        best_score = score
        torch.save({"ema": ema.module.state_dict(), "sig": SIG, "cfg": CFG, "lexicon": LEXICON}, BEST_PATH)
    torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": optimizer.state_dict(),
                "sched": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                "epochs": epochs_total, "sig": SIG, "best": list(best_score)}, LAST_PATH)

print("=" * 110)
print(f"Best quick-validation fused accuracy: {best_score[0]:.1%}  ->  {BEST_PATH}")

## 8. Calibrate on validation, report on the untouched test set

In [ ]:
net = build_model()
net.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=False)["ema"])
net.eval()

print("Calibrating on the full validation set ...")
val_res = collect(net, val_split, val_split.order())
CAL = calibrate(val_res, val_split.stats)
with open(CAL_PATH, "w") as f:
    json.dump(CAL, f, indent=1)
print(f"-> alpha {CAL['alpha']}, temperature {CAL['temperature']:.2f}, confidence threshold {CAL['conf_threshold']:.3f}, "
      f"gate {CAL['gate']}")

# ── test report ──
test_res = collect(net, test_split, test_split.order())
ids = test_res["ids"]
fused = fuse(test_res, CAL["alpha"])
probs = torch.softmax(fused / CAL["temperature"], dim=1)
conf, pred = probs.max(1)
is_noise = ids == NOISE_ID
speech = ~is_noise

gate_bad = ((test_split.stats[:, STAT_NAMES.index("loudness_db")] < CAL["gate"]["min_loudness_db"]) |
            (test_split.stats[:, STAT_NAMES.index("active_ratio")] < CAL["gate"]["min_active_ratio"]))
emit = (conf >= CAL["conf_threshold"]) & (pred != NOISE_ID) & ~gate_bad       # the model actually outputs a word
right = emit & (pred == ids)
wrong = emit & (pred != ids)

qm = quick_metrics(test_res, CAL["alpha"])
print(f"\nTEST  (speech {int(speech.sum()):,} clips + {int(is_noise.sum()):,} noise/silence clips)")
print(f"  accuracy, no abstention : greedy {qm['greedy']:.1%} | ctc-lexicon {qm['ctc']:.1%} | classifier {qm['cls']:.1%} | fused {qm['fused']:.1%}")
print(f"  with abstention          : answers {emit[speech].float().mean():.1%} of speech clips | "
      f"precision of answers {right.sum().item() / max(emit.sum().item(), 1):.1%}  (target {TARGET_PRECISION:.0%})")
print(f"  wrong words emitted      : {wrong[speech].sum().item()} of {int(speech.sum())} speech clips "
      f"({wrong[speech].float().mean():.2%})  vs {(pred[speech] != ids[speech]).sum().item()} "
      f"({(pred[speech] != ids[speech]).float().mean():.2%}) if it always answered")
print(f"  noise / silence clips    : {1 - emit[is_noise].float().mean():.1%} correctly produced NO word "
      f"(words hallucinated on noise: {emit[is_noise].sum().item()} of {int(is_noise.sum())})")
print(f"  speech wrongly gated as 'no speech': {gate_bad[speech].float().mean():.2%}")

conf_pairs = Counter((LEXICON[t], LEXICON[p]) for t, p in zip(ids.tolist(), pred.tolist()) if t != p)
print("\nMost confused (truth -> predicted):")
for (t, p), n in conf_pairs.most_common(8):
    print(f"  {t:>9} -> {p:<9} x{n}")

## 9. Inference + tone analysis
`analyze()` returns the decision (`accepted` / `uncertain` / `no_speech`), calibrated confidence, alternatives and the
measured tone descriptors. Longer recordings are cropped to their loudest second automatically.
*In a fresh session you only need cells 0, 1, 2 and 5 (definitions) plus this section: the model and calibration are
loaded from `WORK_DIR`.*

In [ ]:
def load_for_inference():
    """Load the best model + calibration from WORK_DIR (works without the feature cache)."""
    global net_infer, CAL_INFER
    ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
    with open(CAL_PATH) as f:
        CAL_INFER = json.load(f)
    set_lexicon([w for w in CAL_INFER["lexicon"] if w != NOISE_TOKEN])
    net_infer = build_model()
    net_infer.load_state_dict(ckpt["ema"])
    net_infer.eval()
    return net_infer


load_for_inference()


def prepare_waveform(waveform, sample_rate=SAMPLE_RATE):
    w = torch.as_tensor(waveform, dtype=torch.float32)
    if w.ndim == 2:
        w = w.mean(0)                                                  # mono
    if sample_rate != SAMPLE_RATE:
        w = torchaudio.functional.resample(w, sample_rate, SAMPLE_RATE)
    if w.numel() > CLIP_SAMPLES:                                       # keep the loudest second
        hop = SAMPLE_RATE // 10
        cs = F.pad(w.pow(2).cumsum(0), (1, 0))
        starts = torch.arange(0, w.numel() - CLIP_SAMPLES + 1, hop)
        w = w[int(starts[(cs[starts + CLIP_SAMPLES] - cs[starts]).argmax()]):][:CLIP_SAMPLES]
    return fix_length(w)


def describe(name, value, ref):
    lo, hi = ref
    return {"loudness_db": ("soft", "normal", "loud"), "f0_mean_hz": ("low", "mid", "high"),
            "centroid_hz": ("dark", "neutral", "bright")}[name][0 if value < lo else 2 if value > hi else 1]


@torch.no_grad()
def analyze(waveform, sample_rate=SAMPLE_RATE, verbose=True):
    cal = CAL_INFER
    wave = prepare_waveform(waveform, sample_rate).to(device)
    feats, stats = frontend(wave[None])
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        ctc_logits, word_logits = net_infer(feats)
    fused = cal["alpha"] * lexicon_ll(ctc_logits) + (1 - cal["alpha"]) * F.log_softmax(word_logits.float(), -1)
    probs = torch.softmax(fused / cal["temperature"], dim=1)[0].cpu()
    top = probs.argsort(descending=True)[:3].tolist()
    s = dict(zip(STAT_NAMES, stats[0].cpu().tolist()))

    gated = s["loudness_db"] < cal["gate"]["min_loudness_db"] or s["active_ratio"] < cal["gate"]["min_active_ratio"]
    word, conf = LEXICON[top[0]], float(probs[top[0]])
    if gated or word == NOISE_TOKEN:
        status, answer = "no_speech", None
    elif conf < cal["conf_threshold"]:
        status, answer = "uncertain", None
    else:
        status, answer = "accepted", word

    ref = cal["tone_ref"]
    slope = s["f0_slope_st_s"]
    tone = {"loudness_db": round(s["loudness_db"], 1), "loudness": describe("loudness_db", s["loudness_db"], ref["loudness_db"]),
            "peak_db": round(s["peak_db"], 1),
            "speech_ms": int(round(s["active_ratio"] * 1000)),
            "voiced_ratio": round(s["voiced_ratio"], 2),
            "pitch_hz": round(s["f0_mean_hz"], 1),
            "pitch": describe("f0_mean_hz", s["f0_mean_hz"], ref["f0_mean_hz"]) if s["f0_mean_hz"] > 0 else "n/a",
            "pitch_range_st": round(s["f0_std_st"], 2),
            "contour": "n/a" if s["f0_mean_hz"] == 0 else ("rising" if slope > 3 else "falling" if slope < -3 else "flat"),
            "pitch_slope_st_s": round(slope, 1),
            "brightness_hz": int(s["centroid_hz"]), "brightness": describe("centroid_hz", s["centroid_hz"], ref["centroid_hz"])}
    result = {"status": status, "word": answer, "best_guess": word, "confidence": round(conf, 3),
              "alternatives": [(LEXICON[i], round(float(probs[i]), 3)) for i in top[1:]], "tone": tone}

    if verbose:
        head = {"accepted": f"'{answer}'", "uncertain": f"uncertain (best guess '{word}')", "no_speech": "no speech detected"}[status]
        print(f"Result  : {head}" + ("" if status == "no_speech" else f" | confidence {conf:.1%}") + f" | {status}")
        if status != "no_speech":
            print("Also    : " + ", ".join(f"'{w}' {p:.1%}" for w, p in result["alternatives"]))
        print(f"Tone    : {tone['loudness']} ({tone['loudness_db']} dBFS) | pitch {tone['pitch']} ({tone['pitch_hz']} Hz, "
              f"range {tone['pitch_range_st']} st, {tone['contour']}) | brightness {tone['brightness']} | "
              f"voiced {tone['voiced_ratio']:.0%} | speech {tone['speech_ms']} ms")
    return result


def plot_analysis(waveform, sample_rate=SAMPLE_RATE):
    wave = prepare_waveform(waveform, sample_rate).to(device)
    r = {k: v[0].cpu() for k, v in frontend.raw(wave[None]).items()}
    t = torch.arange(r["energy_db"].numel()) * HOP_LENGTH / SAMPLE_RATE
    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(torch.arange(wave.numel()) / SAMPLE_RATE, wave.cpu().numpy(), lw=0.6)
    ax[0].set(title="Waveform (volume over time)", ylabel="Amplitude")
    ax[1].imshow(r["mel_db"].numpy(), origin="lower", aspect="auto", cmap="viridis",
                 extent=[0, t[-1].item(), 0, N_MELS])
    ax[1].set(title="Log-Mel spectrogram", ylabel="Mel bin")
    f0 = torch.where(r["voiced"], r["f0"], torch.full_like(r["f0"], float("nan")))
    ax[2].plot(t, f0.numpy(), "o", ms=3, color="tab:red")
    ax[2].set(title="Pitch contour (red, voiced frames) and loudness (dashed)", ylabel="F0 (Hz)", xlabel="Time (s)")
    ax2 = ax[2].twinx()
    ax2.plot(t, r["energy_db"].clamp_min(-80).numpy(), "--", color="tab:gray")
    ax2.set_ylabel("dBFS")
    plt.tight_layout()
    plt.show()

In [ ]:
# Demo: real test clips, a synthetic voice, noise and silence
test_raw = load_split("testing")
for i in random.sample(range(len(test_raw)), 4):
    waveform, sr, label, *_ = test_raw[i]
    print(f"\n--- truth: '{label}' ---")
    analyze(waveform, sr)

print("\n--- synthetic 180 Hz voice-like tone ---")
analyze(harmonic_tone(180))
print("\n--- white noise ---")
analyze(0.2 * torch.randn(SAMPLE_RATE))
print("\n--- silence ---")
analyze(torch.zeros(SAMPLE_RATE))

plot_analysis(waveform, sr)
display(Audio(waveform.numpy(), rate=sr))